## Example 1 — the import, the version and the FFmpeg libraries it is linked to

In [ ]:
import importlib.metadata
import av

print("av", av.__version__, "-- FFmpeg", av.ffmpeg_version_info)
assert av.__version__ == importlib.metadata.version("av")
EXPECTED = {"libavutil", "libavcodec", "libavformat", "libavdevice", "libavfilter", "libswscale", "libswresample"}
assert set(av.library_versions) == EXPECTED, sorted(av.library_versions)
assert all(len(version) == 3 and version[0] > 0 for version in av.library_versions.values()), av.library_versions
for codec in ("ffv1", "flac", "pcm_s16le", "mpeg4", "aac"):
    assert codec in av.codecs_available, f"FFmpeg was built without {codec}"
for container in ("matroska", "mp4", "wav", "ogg", "webm"):
    assert container in av.formats_available, f"FFmpeg was built without the {container} container"
print(f"OK -- linked to {len(av.library_versions)} FFmpeg libraries; {len(av.codecs_available)} codecs and {len(av.formats_available)} containers")

## Example 2 — video: encode five frames losslessly in memory, decode them back bit for bit

In [ ]:
import io
import av
import numpy as np

frames = [((np.add.outer(np.arange(48), np.arange(64)) * (i + 1)) % 256).astype(np.uint8) for i in range(5)]
buffer = io.BytesIO()
with av.open(buffer, mode="w", format="matroska") as container:
    stream = container.add_stream("ffv1", rate=25)  # FFV1 is lossless
    stream.width, stream.height, stream.pix_fmt = 64, 48, "gray"
    for image in frames:
        for packet in stream.encode(av.VideoFrame.from_ndarray(image, format="gray")):
            container.mux(packet)
    for packet in stream.encode():  # flush
        container.mux(packet)
buffer.seek(0)
with av.open(buffer) as container:
    video = container.streams.video[0]
    assert (video.codec_context.name, video.width, video.height) == ("ffv1", 64, 48)
    decoded = [frame.to_ndarray(format="gray") for frame in container.decode(video)]
assert len(decoded) == 5 and all(np.array_equal(a, b) for a, b in zip(frames, decoded)), "a decoded frame differs"
print(f"OK -- 5 frames of 64x48 through FFV1 in a Matroska container ({buffer.getbuffer().nbytes} bytes) come back identical")

## Example 3 — audio: a tone to WAV and back, then resampled from 16 kHz to 8 kHz

In [ ]:
import io
import av
import numpy as np

rate = 16000
tone = (np.sin(2 * np.pi * 440 * np.arange(rate) / rate) * 20000).astype(np.int16).reshape(1, -1)  # one second, mono
buffer = io.BytesIO()
with av.open(buffer, mode="w", format="wav") as container:
    stream = container.add_stream("pcm_s16le", rate=rate, layout="mono")
    frame = av.AudioFrame.from_ndarray(tone, format="s16", layout="mono")
    frame.sample_rate = rate
    for packet in stream.encode(frame) + stream.encode():
        container.mux(packet)
buffer.seek(0)
with av.open(buffer) as container:
    audio = container.streams.audio[0]
    assert (audio.codec_context.name, audio.sample_rate, audio.codec_context.channels) == ("pcm_s16le", rate, 1)
    samples = np.concatenate([f.to_ndarray() for f in container.decode(audio)], axis=1)
assert np.array_equal(samples, tone), "the decoded samples differ from the tone"
resampler = av.AudioResampler(format="s16", layout="mono", rate=8000)
halved = resampler.resample(frame) + resampler.resample(None)
count = sum(f.samples for f in halved)
assert abs(count - 8000) <= 64 and all(f.sample_rate == 8000 for f in halved), count
print(f"OK -- {tone.shape[1]} samples survive a WAV round trip exactly; resampling to 8 kHz gives {count}")

## Example 4 — a filter graph: flip a frame and scale it

In [ ]:
import av
import av.filter
import numpy as np

image = (np.add.outer(np.arange(32), np.arange(48) * 3) % 256).astype(np.uint8)
frame = av.VideoFrame.from_ndarray(image, format="gray")
graph = av.filter.Graph()
source = graph.add_buffer(width=48, height=32, format="gray", time_base=frame.time_base or "1/25")
flip = graph.add("hflip")
sink = graph.add("buffersink")
source.link_to(flip)
flip.link_to(sink)
graph.configure()
graph.push(frame)
flipped = graph.pull().to_ndarray(format="gray")
assert np.array_equal(flipped, image[:, ::-1]), "hflip did not mirror the frame"
doubled = frame.reformat(width=96, height=64, format="gray")  # libswscale
assert (doubled.width, doubled.height) == (96, 64)
rgb = frame.to_ndarray(format="rgb24")  # a pixel-format conversion: gray to RGB keeps the three channels equal
assert rgb.shape == (32, 48, 3) and np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 1], rgb[..., 2])
print("OK -- hflip mirrors the frame exactly; libswscale scales it to 96x64 and converts it to RGB")

## Example 5 — bad input raises PyAV's own errors, never a crash

In [ ]:
import io, os, tempfile
import av, av.error

refused = {}
try:
    av.open(io.BytesIO(b"this is not a media file" * 64))
except av.error.InvalidDataError as error:
    refused["garbage"] = type(error).__name__
else:
    raise AssertionError("garbage was opened as a media file")
try:
    av.open(os.path.join(tempfile.mkdtemp(prefix="av_nb_"), "missing.mp4"))
except av.error.FileNotFoundError as error:
    assert isinstance(error, FileNotFoundError)  # also the built-in class, so ordinary handlers catch it
    refused["missing file"] = type(error).__name__
else:
    raise AssertionError("a missing file was opened")
try:
    av.codec.Codec("no_such_codec", "w")
except av.codec.codec.UnknownCodecError as error:
    refused["unknown codec"] = type(error).__name__
else:
    raise AssertionError("an unknown codec name was accepted")
print("OK -- refused:", refused)